# 🌊 Week 1, Day 3 — May 21, 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, os, datetime, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {{
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

## Step 1: Load Filtered Datasets from /data/processed/

In [ ]:
df_plastic  = pd.read_csv(DIRS['processed'] + '/plastic_filtered.csv', parse_dates=['Date'])
df_species  = pd.read_csv(DIRS['processed'] + '/species_filtered.csv')
df_climate  = pd.read_csv(DIRS['processed'] + '/climate_flagged.csv',  parse_dates=['Date'])
df_temp     = pd.read_csv(DIRS['processed'] + '/global_temp_filtered.csv', parse_dates=['dt'])
df_co2      = pd.read_csv(DIRS['processed'] + '/sea_level_co2_filtered.csv', parse_dates=['date'])
print('All processed datasets loaded ✅')
print(f'  plastic : {df_plastic.shape}')
print(f'  species : {df_species.shape}')
print(f'  climate : {df_climate.shape}')
print(f'  temp    : {df_temp.shape}')
print(f'  co2     : {df_co2.shape}')

## Step 2: Schema Review — ocean_plastic_pollution_data

In [ ]:
print('=== ocean_plastic_pollution_data SCHEMA ===')
print(df_plastic.info())
print()
print('Dtypes:')
print(df_plastic.dtypes)
print()
print('Sample rows:')
print(df_plastic.head(3).to_string())

In [ ]:
# Check for zero-variance columns (constant values across all rows)
print('Zero-variance check (numeric cols):')
num_cols = df_plastic.select_dtypes(include='number').columns
for col in num_cols:
    if df_plastic[col].nunique() == 1:
        print(f'  ⚠️  {col} is CONSTANT: {df_plastic[col].iloc[0]}')
    else:
        print(f'  ✅  {col} — {df_plastic[col].nunique()} unique values  (range: {df_plastic[col].min():.2f} – {df_plastic[col].max():.2f})')

print()
print('Categorical columns unique values:')
for col in df_plastic.select_dtypes(include='object').columns:
    print(f'  {col}: {df_plastic[col].unique().tolist()}')

## Step 3: Schema Review — india_cms_final_master_enriched

In [ ]:
print('=== india_cms_final_master_enriched SCHEMA ===')
print(df_species.info())

In [ ]:
# Flag structural issues found in species dataset
issues = []

# 1. year is float64, should be int (after cleaning anomalies)
print(f'year dtype: {df_species["year"].dtype}  → should be int64 after cleaning')
print(f'year range: {df_species["year"].min()} to {df_species["year"].max()}')
print(f'⚠️  Anomalous years outside 2015-2026: {len(df_species[~df_species["year"].between(2015,2026)])}')
issues.append('year: float64 dtype, anomalous values outside 2015-2026')

# 2. priority_group — 87% null but NOT zero-variance (has 2 values)
print(f'\npriority_group unique values: {df_species["priority_group"].dropna().unique().tolist()}')
print(f'priority_group null rate: {df_species["priority_group"].isnull().mean()*100:.1f}%')
print('⚠️  87% null — will DROP this column in Week 2')
issues.append('priority_group: 87% null rate — drop candidate')

# 3. observation_type — 12 different labels, some look like they need normalization
print(f'\nobservation_type values: {df_species["observation_type"].unique().tolist()}')
print('⚠️  12 inconsistent labels — normalize to 3 categories in Week 2')
issues.append('observation_type: 12 inconsistent labels')

# 4. data_source — check if NBN-Atlas-UK records belong in an India project
nbn = df_species[df_species['data_source'] == 'NBN-Atlas-UK']
print(f'\nNBN-Atlas-UK records: {len(nbn)} (these are UK records — review for exclusion)')
issues.append('data_source: NBN-Atlas-UK has 630 UK records in an India dataset')

print(f'\nTotal structural issues flagged: {len(issues)}')
for i, iss in enumerate(issues, 1):
    print(f'  Issue #{i}: {iss}')

## Step 4: Schema Review — realistic_ocean_climate_dataset

In [ ]:
print('=== realistic_ocean_climate_dataset SCHEMA ===')
print(df_climate.info())
print()
print('Unique Locations:', df_climate['Location'].unique().tolist())
print('SST range (°C):', df_climate['SST (°C)'].min(), 'to', df_climate['SST (°C)'].max())
print('pH range:', df_climate['pH Level'].min(), 'to', df_climate['pH Level'].max())
print('Marine Heatwave dtype:', df_climate['Marine Heatwave'].dtype, '— bool, correct ✅')
print()
print('Bleaching Severity values:')
print(df_climate['Bleaching Severity'].value_counts(dropna=False))

## Step 5: Naming Inconsistency Audit Across Datasets

In [ ]:
print('NAMING INCONSISTENCY AUDIT')
print('='*50)

# Plastic types — full verbose names
print('Plastic_Type values (verbose):')
for pt in sorted(df_plastic['Plastic_Type'].unique()):
    print(f'  {pt}')
print('→ Will shorten to: PET, PE, PS, PP, PVC in Week 2')

# Region label mismatch (from Day 2 finding)
print('\nRegion labels in plastic data (unreliable):')
print(df_plastic['Region'].value_counts().to_dict())
print('→ Confirmed: Region labels do NOT match coordinates (Issue from Day 2)')

# Species observation_type — inconsistent casing and format
print('\nobservation_type normalization map:')
obs_map = {
    'Scientific Record': 'Scientific',
    'PRESERVED_SPECIMEN': 'Scientific',
    'OCCURRENCE': 'Occurrence',
    'HUMAN_OBSERVATION': 'Human_Observation',
    'MATERIAL_CITATION': 'Scientific',
    'MATERIAL_SAMPLE': 'Scientific',
    'MACHINE_OBSERVATION': 'Machine_Observation',
    'MARINE_OBS': 'Human_Observation',
    'UK_RECORD': 'Human_Observation',
    'REPTILE_CITIZEN': 'Citizen_Science',
    'CITIZEN_SCIENCE': 'Citizen_Science',
    'OBSERVATION': 'Human_Observation'
}
for old, new in obs_map.items():
    print(f'  "{old}" → "{new}"')

## Step 6: Save Schema Audit Report

In [ ]:
report_path = DIRS['metadata'] + '/schema_audit_may21.txt'

with open(report_path, 'w') as f:
    f.write('HCLTech Internship — Schema Audit Report\n')
    f.write(f'Date: May 21, 2026\n')
    f.write('='*60 + '\n\n')
    f.write('DATASET SHAPES:\n')
    for name, df in [('ocean_plastic',df_plastic),('india_species',df_species),
                     ('ocean_climate',df_climate),('global_temp',df_temp),('sea_level_co2',df_co2)]:
        f.write(f'  {name:<20}: {df.shape[0]:>6} rows × {df.shape[1]:>2} cols\n')
    f.write('\nISSUES FLAGGED FOR WEEK 2 CLEANING:\n')
    cleanup_items = [
        'ocean_plastic: Plastic_Type uses verbose names — shorten to abbreviations',
        'ocean_plastic: Region label is unreliable — do not use as spatial key',
        'india_species: year is float64 with anomalies — cast to int, filter 2015-2026',
        'india_species: priority_group is 87% null — DROP column',
        'india_species: observation_type has 12 inconsistent labels — normalize to 5',
        'india_species: NBN-Atlas-UK source has 630 UK records — review for exclusion',
        'ocean_climate: Bleaching Severity has 150 nulls — fill with Unknown',
        'global_temp: LandMaxTemperature is 37.6% null — only use LandAndOcean column',
    ]
    for i, item in enumerate(cleanup_items, 1):
        f.write(f'  [{i}] {item}\n')

print(f'Schema audit saved: {report_path} ✅')
with open(report_path) as f:
    print(f.read())